# TS-SatFire FP - Label Audit (v2, paper-correct splits)

Corrections vs v1:
1. VAL_IDS = 15 fires per paper code (adds 23301962 and 22713339)
2. TEST set = US_2021 fires only (paper's 2021 CSV), NOT named fires
3. Probes FirePred/ directory to confirm band-8-diff == pre-computed FP label

In [1]:
# ============================================================
# CELL 1 - Probe FirePred directory structure
# ============================================================
import os, glob, rasterio
import numpy as np

DATA_ROOT = "/kaggle/input/datasets/z789456sx/ts-satfire/ts-satfire"
probe_fire = "20778186"
fpdir = os.path.join(DATA_ROOT, probe_fire, "FirePred")

fp_tifs = sorted(glob.glob(os.path.join(fpdir, "*.tif")))
print(f"FirePred files for {probe_fire}: {len(fp_tifs)}")
if fp_tifs:
    with rasterio.open(fp_tifs[0]) as src:
        print(f"  first file: {os.path.basename(fp_tifs[0])}")
        print(f"  shape: {src.height} x {src.width}, bands: {src.count}, dtype: {src.dtypes[0]}")
        # Inspect each band
        for b in range(1, src.count + 1):
            arr = src.read(b).astype(np.float32)
            finite = np.isfinite(arr).sum()
            nan = np.isnan(arr).sum()
            uniq = np.unique(arr[np.isfinite(arr)])[:10] if finite > 0 else []
            print(f"  band {b}: finite={finite}, nan={nan}, unique_samples={uniq}")

# Compare with VIIRS_Day band 8 diff for same day
day_tifs = sorted(glob.glob(os.path.join(DATA_ROOT, probe_fire, "VIIRS_Day", "*.tif")))
if len(day_tifs) >= 2 and fp_tifs:
    with rasterio.open(day_tifs[0]) as src:
        b8_d0 = src.read(8).astype(np.float32)
    with rasterio.open(day_tifs[1]) as src:
        b8_d1 = src.read(8).astype(np.float32)
    diff = ((~np.isnan(b8_d1)) & np.isnan(b8_d0)).sum()
    print(f"\nBand-8 diff (day1 finite AND day0 NaN) = {diff} pixels")
    with rasterio.open(fp_tifs[1]) as src:
        fp_label = src.read(src.count).astype(np.float32)  # assume last band is label
    fp_pos = int((fp_label > 0).sum())
    print(f"FirePred file for same day (band {src.count}, >0 count) = {fp_pos} pixels")

FirePred files for 20778186: 57
  first file: 2017-07-17_FirePred.tif
  shape: 447 x 447, bands: 19, dtype: float32
  band 1: finite=199797, nan=12, unique_samples=[-1081. -1047. -1037.  -952.  -849.  -794.  -785.  -763.  -762.  -735.]
  band 2: finite=199797, nan=12, unique_samples=[-252. -157. -138. -115. -112. -106. -105. -100.  -97.  -95.]
  band 3: finite=199809, nan=0, unique_samples=[0.]
  band 4: finite=199809, nan=0, unique_samples=[1.9 2.  2.1 2.2 2.3 2.4 2.5 2.6 2.7 2.8]
  band 5: finite=199809, nan=0, unique_samples=[254. 255. 256. 257. 258. 259. 260. 261. 262. 263.]
  band 6: finite=199809, nan=0, unique_samples=[271.6 271.8 272.8 273.4 273.5 273.6 273.7 273.8 273.9 274.2]
  band 7: finite=199809, nan=0, unique_samples=[290.5 290.6 290.7 290.9 291.  291.2 291.4 291.5 291.6 291.7]
  band 8: finite=199809, nan=0, unique_samples=[33. 37. 38. 39. 40. 41. 42. 43. 44. 45.]
  band 9: finite=199809, nan=0, unique_samples=[0.00416 0.00417 0.00421 0.00422 0.00423 0.00426 0.00427 0.0

In [2]:
# ============================================================
# CELL 2 - FP audit with PAPER-CORRECT splits
# ============================================================
import os, glob, rasterio
import numpy as np

DATA_ROOT = "/kaggle/input/datasets/z789456sx/ts-satfire/ts-satfire"
TS = 2
PATCH = 256
BA_BAND = 8

# Paper's FP val_ids (from dataset_gen_pred.py - 15 fires)
FP_VAL_IDS = {"20568194", "20701026", "20562846", "20700973", "24462610",
              "24462788", "24462753", "24103571", "21998313", "21751303",
              "22141596", "21999381", "23301962", "22712904", "22713339"}

all_ids = sorted(os.listdir(DATA_ROOT))
numeric_ids = [d for d in all_ids if d.isdigit()]
us2021_ids  = [d for d in all_ids if d.startswith("US_2021")]

fp_train_ids = [d for d in numeric_ids if d not in FP_VAL_IDS]
fp_val_ids   = [d for d in numeric_ids if d in FP_VAL_IDS]
fp_test_ids  = sorted(us2021_ids)

# Check which expected val IDs are missing
missing_val = FP_VAL_IDS - set(numeric_ids)
print(f"FP train: {len(fp_train_ids)} fires")
print(f"FP val:   {len(fp_val_ids)} fires (expected 15)")
print(f"FP test:  {len(fp_test_ids)} fires (US_2021)")
if missing_val:
    print(f"WARNING: val IDs missing from dataset: {missing_val}")


def audit_fp(fid):
    fdir = os.path.join(DATA_ROOT, fid)
    day_tifs = sorted(glob.glob(os.path.join(fdir, "VIIRS_Day", "*.tif")))
    fp_tifs  = sorted(glob.glob(os.path.join(fdir, "FirePred", "*.tif")))
    n = len(day_tifs)
    res = {"id": fid, "days": n, "fp_files": len(fp_tifs),
           "days_ba_ok": 0, "windows_total": 0, "windows_pair_ok": 0,
           "windows_progression": 0, "total_new_px": 0, "max_new_px": 0,
           "status": ""}
    if n < TS + 1:
        res["status"] = "TOO_FEW_DAYS"; return res

    with rasterio.open(day_tifs[0]) as src:
        H, W = src.height, src.width
    r0 = max(0, (H - PATCH) // 2); c0 = max(0, (W - PATCH) // 2)
    ph = min(PATCH, H); pw = min(PATCH, W)

    ba_masks = [None] * n
    for i, tif in enumerate(day_tifs):
        with rasterio.open(tif) as src:
            if src.count < BA_BAND: continue
            b8 = src.read(BA_BAND).astype(np.float32)
        if np.isnan(b8).sum() == b8.size: continue
        mask = (~np.isnan(b8)).astype(np.uint8)
        ba_masks[i] = mask[r0:r0+ph, c0:c0+pw]
        res["days_ba_ok"] += 1

    for t in range(n - TS):
        res["windows_total"] += 1
        last = ba_masks[t + TS - 1]; nxt = ba_masks[t + TS]
        if last is None or nxt is None: continue
        res["windows_pair_ok"] += 1
        new_burn = ((nxt == 1) & (last == 0)).astype(np.uint8)
        npx = int(new_burn.sum())
        if npx > 0:
            res["windows_progression"] += 1
            res["total_new_px"] += npx
            if npx > res["max_new_px"]: res["max_new_px"] = npx

    if res["windows_pair_ok"] == 0: res["status"] = "NO_VALID_PAIRS"
    elif res["windows_progression"] == 0: res["status"] = "NO_PROGRESSION"
    elif res["windows_pair_ok"] < max(1, 0.3 * res["windows_total"]): res["status"] = "MOSTLY_BAD"
    else: res["status"] = "OK"
    return res


def run_split(ids, label):
    print(f"\nAuditing {label} ({len(ids)} fires)...")
    out = []
    for i, fid in enumerate(ids):
        out.append(audit_fp(fid))
        if (i + 1) % 20 == 0 or (i + 1) == len(ids):
            print(f"\r  {i+1}/{len(ids)}", end="", flush=True)
    print()
    return out

train_res = run_split(fp_train_ids, "TRAIN")
val_res   = run_split(fp_val_ids,   "VAL")
test_res  = run_split(fp_test_ids,  "TEST (US_2021)")

FP train: 137 fires
FP val:   14 fires (expected 15)
FP test:  24 fires (US_2021)

Auditing TRAIN (137 fires)...
  137/137

Auditing VAL (14 fires)...
  14/14

Auditing TEST (US_2021) (24 fires)...
  24/24


In [3]:
# ============================================================
# CELL 3 - Summary + exclusion lists
# ============================================================
def summarize(results, label):
    n = len(results)
    ok      = [r for r in results if r["status"] == "OK"]
    mostly  = [r for r in results if r["status"] == "MOSTLY_BAD"]
    no_pair = [r for r in results if r["status"] == "NO_VALID_PAIRS"]
    no_prog = [r for r in results if r["status"] == "NO_PROGRESSION"]
    too_few = [r for r in results if r["status"] == "TOO_FEW_DAYS"]
    tw = sum(r["windows_total"] for r in results)
    tp = sum(r["windows_pair_ok"] for r in results)
    tprog = sum(r["windows_progression"] for r in results)
    tpx = sum(r["total_new_px"] for r in results)

    print(f"\n{'='*72}\n{label}  ({n} fires)\n{'='*72}")
    print(f"  OK:             {len(ok):3d} ({len(ok)/max(n,1)*100:.0f}%)")
    print(f"  MOSTLY_BAD:     {len(mostly):3d}")
    print(f"  NO_PROGRESSION: {len(no_prog):3d}")
    print(f"  NO_VALID_PAIRS: {len(no_pair):3d}")
    print(f"  TOO_FEW_DAYS:   {len(too_few):3d}")
    print(f"  Total windows:          {tw}")
    print(f"  Windows with valid pair: {tp} ({tp/max(tw,1)*100:.1f}%)")
    print(f"  Windows with progress:   {tprog} ({tprog/max(tw,1)*100:.1f}%)")
    print(f"  Total new-burn px:       {tpx:,}")

    excluded = mostly + no_pair + no_prog + too_few
    if excluded:
        print(f"\n  EXCLUDE ({len(excluded)}):")
        for r in sorted(excluded, key=lambda x: x["id"]):
            print(f"    {r['id']:<30s} {r['status']:<18s} "
                  f"pair={r['windows_pair_ok']}/{r['windows_total']}  "
                  f"prog={r['windows_progression']}  px={r['total_new_px']}")
    return [r["id"] for r in excluded]

ex_train = summarize(train_res, "TRAIN")
ex_val   = summarize(val_res,   "VAL")
ex_test  = summarize(test_res,  "TEST (US_2021)")

print("\n\n" + "="*72)
print("FP EXCLUSION LISTS")
print("="*72)
for name, lst in [("FP_TRAIN_EXCLUDE", ex_train),
                  ("FP_VAL_EXCLUDE", ex_val),
                  ("FP_TEST_EXCLUDE", ex_test)]:
    print(f"\n{name} = [")
    for x in sorted(lst): print(f'    "{x}",')
    print("]")


TRAIN  (137 fires)
  OK:              89 (65%)
  MOSTLY_BAD:       5
  NO_PROGRESSION:   3
  NO_VALID_PAIRS:  27
  TOO_FEW_DAYS:    13
  Total windows:          2006
  Windows with valid pair: 1395 (69.5%)
  Windows with progress:   1188 (59.2%)
  Total new-burn px:       648,829

  EXCLUDE (48):
    20702159                       NO_VALID_PAIRS     pair=0/25  prog=0  px=0
    20777134                       MOSTLY_BAD         pair=10/49  prog=6  px=3363
    20777152                       MOSTLY_BAD         pair=3/15  prog=1  px=1330
    20777181                       NO_VALID_PAIRS     pair=0/9  prog=0  px=0
    20777195                       NO_VALID_PAIRS     pair=0/12  prog=0  px=0
    20777203                       NO_VALID_PAIRS     pair=0/6  prog=0  px=0
    20777207                       TOO_FEW_DAYS       pair=0/0  prog=0  px=0
    20777386                       TOO_FEW_DAYS       pair=0/0  prog=0  px=0
    20777397                       NO_VALID_PAIRS     pair=0/12  prog=0  p